This notebook is to analyse and visualize the aggregatred results from all agents and LLM judges

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import numpy as np
import matplotlib.ticker as ticker
from matplotlib.patches import Patch


In [ ]:
df = pd.read_csv("./result.csv")

### Average document lengths for all agents and models

In [ ]:

avg_df = df.groupby('agent')[['length', 'ground_truth_length','total_ref']].mean().reset_index()

print(avg_df)

### Average scores by difficulty

In [ ]:
avg_scores = score_by_difficulty = (
    df.groupby(['agent', 'judge'])['score']
    .agg(['mean', 'std'])
    .reset_index()
    .round({'mean': 2, 'std': 2})
)
avg_scores

In [ ]:

pd.set_option('display.max_rows', 10)  

score_by_difficulty = (
    df.groupby(['agent', 'difficulty', 'judge'])['score']
    .agg(['mean', 'std'])
    .reset_index()
    .round({'mean': 2, 'std': 2})
)

score_by_difficulty

### Average scores by criteria

In [ ]:

score_by_criteria = (
    df.groupby(['agent', 'judge'])[['accuracy', 'clarity', 'structure', 'reference']]
    .agg(['mean', 'std'])
    .round(2)
    .reset_index()
)




In [ ]:
score_by_criteria

In [ ]:

pd.reset_option('display.max_rows')


In [ ]:
df[df["task"] == "task-17"]

In [ ]:

df['agent'] = df['agent'].replace({
    'gemini': 'gemini-cli',
    'claude': 'claude-code'
})


In [ ]:
df["difficulty"] = df["difficulty"].astype(int)

### Plot score by difficulty

In [ ]:

plt.figure(figsize=(16, 10))
sns.lineplot(
    data=df,
    x='difficulty',
    y='score',
    hue='agent',
    style='agent',
    markers=True,
    dashes=False
)

# Ensure no gaps in difficulty levels
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(1))

# Remove title and axis legends
plt.title('')
plt.xlabel('')
plt.ylabel('')

# Legend with no title and all agents/models in one line
plt.legend(title="Agent/Model", loc='upper center', bbox_to_anchor=(0.5, 1.10), ncol=len(df['agent'].unique()))

plt.tight_layout()
plt.show()


### Scores by criteria

In [ ]:


plt.rcParams.update({
    'font.size': 15,      # Global font size
    'font.weight': 'bold' # Global font weight
})

# Desired order
desired_order = ['codex', 'gemini-cli', 'claude-code', 'gpt-5', 'gemini-2.5-pro', 'sonnet-4.5']

# Group by agent and average across all judges
avg_scores = df.groupby('agent')[['accuracy', 'clarity', 'structure', 'reference']].mean().reindex(desired_order).dropna().reset_index()

# Plot setup
metrics = ['accuracy', 'clarity', 'structure', 'reference']
x = np.arange(len(metrics))
num_agents = len(avg_scores)
bar_width = 0.8 / num_agents

fig, ax = plt.subplots(figsize=(18, 8))

# Colors and hatches
# colors = ['#c6dbef', '#9ecae1', '#6baed6', '#3182bd', '#08519c', '#bdd7e7']

# colors = [
#         "#c6dbef", "#9ecae1", "#6baed6",
#         "#c6dbef", "#9ecae1", "#6baed6"
#     ]


colors = [
    "#c7e9c0",  # Faded green
    "#9ecae1",  # Faded blue
    "#fdd0a2",  # Faded orange
    "#c7e9c0",  # Faded green
    "#9ecae1",  # Faded blue
    "#fdd0a2",  # Faded orange
]


hatch_patterns = ['', '/']  # Only two patterns: Agent and LLM

# Define which are agents vs LLMs
agent_names = ['codex', 'gemini-cli', 'claude-code']
llm_names = ['gpt-5', 'gemini-2.5-pro', 'sonnet-4.5']

# Plot bars
for i, row in avg_scores.iterrows():
    name = row['agent']
    scores = row[metrics].values
    offset = (i - num_agents / 2) * bar_width + bar_width / 2
    hatch = hatch_patterns[0] if name in agent_names else hatch_patterns[1]
    bars = ax.bar(x + offset, scores, width=bar_width,
                  label=name,
                  color=colors[i % len(colors)],
                  hatch=hatch,
                  edgecolor='black')

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.05,
                f'{height:.2f}', ha='center', va='bottom', fontsize=12)

# Axis setup
ax.set_xticks(x)
ax.set_xticklabels([m.capitalize() for m in metrics])
ax.set_ylabel('')
# ax.set_title('Average Scores by Agent and LLM Across All Judges')
ax.set_ylim(0, 5)

# Agent legend (top center)
agent_legend = ax.legend(title="Agent/LLM", loc='lower center', bbox_to_anchor=(0.5, 0.98),
                         ncol=num_agents, frameon=False, bbox_transform=fig.transFigure)
fig.add_artist(agent_legend)

# Hatch pattern legend (top right)
hatch_legend_elements = [
    Patch(facecolor='white', edgecolor='black', hatch='', label='Agent'),
    Patch(facecolor='white', edgecolor='black', hatch='/', label='LLM')
]
ax.legend(handles=hatch_legend_elements, loc='upper right', frameon=False, title="")

fig.subplots_adjust(top=0.85)
plt.tight_layout()
plt.show()


### Average scores by judge

In [ ]:
avg_score = df.groupby(['agent',"judge"])[['score']].agg(['mean','std']).reset_index()

# Convert to LaTeX
avg_score